# 🚀 Модуль 4. Урок 2. Дедупликация и нормализация строковых полей через RegEx

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Enlyl/data-analyst-2027/blob/main/course/module_04_data_cleaning/lesson_02_duplicates_string_cleaning_regex.ipynb)

## 📌 Бизнес-контекст и методология
Пользовательский ввод полон опечаток: 'Москва', ' москва ', 'г. Москва', 'МСК'. Аналитик обязан приводить строки к каноническому виду.

In [1]:
# 🚀 Универсальная инициализация окружения и данных
import os
import urllib.request
import sqlite3
import pandas as pd
import numpy as np

def ensure_dataset(filename, subfolder=""):
    """Автоматически находит датасет локально или скачивает с GitHub репозитория курса."""
    if os.path.exists(filename):
        return filename
    candidates = [
        os.path.join("course", "datasets", subfolder, filename),
        os.path.join("course", "datasets", filename),
        os.path.join("..", "datasets", subfolder, filename),
        os.path.join("..", "datasets", filename),
        os.path.join("..", "..", "course", "datasets", subfolder, filename),
        os.path.join("datasets", subfolder, filename),
        os.path.join("datasets", filename),
    ]
    for c in candidates:
        if os.path.exists(c):
            return c
    raw_base = "https://raw.githubusercontent.com/Enlyl/data-analyst-2027/main/course/datasets"
    url = f"{raw_base}/{subfolder}/{filename}" if subfolder else f"{raw_base}/{filename}"
    print(f"📥 Загрузка {filename} из репозитория курса...")
    urllib.request.urlretrieve(url, filename)
    print(f"✅ Датасет {filename} успешно готов!")
    return filename

# Загрузка ключевых датасетов
customers_csv = ensure_dataset("customers.csv", subfolder="raw")
orders_csv = ensure_dataset("orders.csv", subfolder="raw")
saas_csv = ensure_dataset("saas_users.csv", subfolder="raw")
print("🟢 Окружение готово к анализу данных.")


🟢 Окружение готово к анализу данных.


## 🔍 Разбор механики и демонстрация на реальных данных

In [2]:
import pandas as pd
import re

dirty_cities = pd.Series([
    "  Москва ", "москва", "г. Москва", "Санкт-Петербург", "СПб", "спб "
])

def clean_city_name(raw):
    s = str(raw).strip().lower()
    s = re.sub(r'г\.\s*', '', s)
    if 'москв' in s:
        return 'Москва'
    elif 'петербург' in s or 'спб' in s:
        return 'Санкт-Петербург'
    return s.title()

cleaned = dirty_cities.apply(clean_city_name)
print(cleaned)


0             Москва
1             Москва
2             Москва
3    Санкт-Петербург
4    Санкт-Петербург
5    Санкт-Петербург
dtype: str


### 🎯 Практическое задание 1: Нормализация номеров телефонов
Напишите функцию очистки номеров телефонов от скобок, пробелов и дефисов, приводящую их к формату 79991234567.

In [3]:
### ВАШ КОД ЗДЕСЬ
raw_phones = pd.Series(["+7 (999) 123-45-67", "8 999 1234567", "79991234567", "+7(999)1234567"])

def normalize_phone(phone):
    digits = re.sub(r'\D', '', str(phone))
    if digits.startswith('8'):
        digits = '7' + digits[1:]
    return digits

clean_phones = raw_phones.apply(normalize_phone)
print(clean_phones)

assert (clean_phones == '79991234567').all(), "Все номера должны быть приведены к каноническому виду"
print("✅ Тест успешно пройден: RegEx очистка работает безошибочно!")


0    79991234567
1    79991234567
2    79991234567
3    79991234567
dtype: str
✅ Тест успешно пройден: RegEx очистка работает безошибочно!


### 💡 Дедупликация в Pandas
`df.drop_duplicates(subset=['user_id'], keep='last')` — сохраняет самое актуальное состояние записи.